In [ ]:
import os
import json
import glob
from datetime import datetime
from pathlib import Path

import numpy as np
import moderngl
from PIL import Image

# =========================================================
# Konfiguration
# =========================================================
N_SCENES = 20
IMAGES_PER_SCENE = 10

# Geometrin i varje scen
N_CYLINDERS_RANGE = (8, 14)
RADIUS_RANGE = (0.9, 2.0)
HEIGHT = 50.0
SPACING = 3.0
MAX_ATTEMPTS = 4000

# Rendering
W, H = 512, 512

# Scenens storlek i world units
BASE_Y_SPAN = 28.0   # Om du vill ha ännu mer utspridning i världen, höj också BASE_Y_SPAN.
CROP_MARGIN = 1.25   # >1.0 => cylindrarna täcker lite mer än cropen

Y_SPAN = BASE_Y_SPAN * CROP_MARGIN
X_SPAN = Y_SPAN * (W / H)

X_RANGE = (-X_SPAN / 2, X_SPAN / 2)
Y_RANGE = (-Y_SPAN / 2, Y_SPAN / 2)

# Kamera
MIN_CAM_RADIUS_FACTOR = 0.9
MAX_CAM_RADIUS_FACTOR = 1.3
ELEV_RANGE = (15, 55)

# Bakgrunder
BACKGROUND_DIR = Path("backgrounds")

# Output
run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")
BASE_DIR = Path("dataset") / run_name
BASE_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng()

# =========================================================
# Hjälpfunktioner
# =========================================================
def overlaps(x0, y0, r0, cylinders, spacing):
    for x1, y1, r1, _ in cylinders:
        if np.hypot(x0 - x1, y0 - y1) < (r0 + r1 + spacing):
            return True
    return False

def generate_scene_cylinders(rng):
    n_cylinders = int(rng.integers(N_CYLINDERS_RANGE[0], N_CYLINDERS_RANGE[1] + 1))
    height = HEIGHT

    cylinders = []
    for _ in range(n_cylinders):
        placed = False
        for _attempt in range(MAX_ATTEMPTS):
            r = float(rng.uniform(*RADIUS_RANGE))
            x0 = float(rng.uniform(X_RANGE[0] + r, X_RANGE[1] - r))
            y0 = float(rng.uniform(Y_RANGE[0] + r, Y_RANGE[1] - r))

            if not overlaps(x0, y0, r, cylinders, SPACING):
                cylinders.append((x0, y0, r, height))
                placed = True
                break

        if not placed:
            print("Varning: kunde inte placera alla cylindrar i en scen.")
            break

    scene_data = {
        "scene": {
            "x_range": list(X_RANGE),
            "y_range": list(Y_RANGE),
            "height": height,
            "spacing": SPACING
        },
        "cylinders": [
            {
                "id": i,
                "x": float(x0),
                "y": float(y0),
                "radius": float(r),
                "height": float(h)
            }
            for i, (x0, y0, r, h) in enumerate(cylinders)
        ]
    }

    return cylinders, scene_data

def perspective(fovy_deg, aspect, near, far):
    f = 1.0 / np.tan(np.radians(fovy_deg) / 2.0)
    m = np.zeros((4, 4), dtype=np.float32)
    m[0, 0] = f / aspect
    m[1, 1] = f
    m[2, 2] = (far + near) / (near - far)
    m[2, 3] = (2 * far * near) / (near - far)
    m[3, 2] = -1.0
    return m

def look_at(eye, target, up):
    eye = np.asarray(eye, dtype=np.float32)
    target = np.asarray(target, dtype=np.float32)
    up = np.asarray(up, dtype=np.float32)

    f = target - eye
    f = f / np.linalg.norm(f)

    s = np.cross(f, up)
    s = s / np.linalg.norm(s)

    u = np.cross(s, f)

    m = np.eye(4, dtype=np.float32)
    m[0, :3] = s
    m[1, :3] = u
    m[2, :3] = -f
    m[0, 3] = -np.dot(s, eye)
    m[1, 3] = -np.dot(u, eye)
    m[2, 3] = np.dot(f, eye)
    return m

def camera_position(radius, elev_deg, azim_deg):
    elev = np.radians(elev_deg)
    azim = np.radians(azim_deg)
    x = radius * np.cos(elev) * np.cos(azim)
    y = radius * np.cos(elev) * np.sin(azim)
    z = radius * np.sin(elev)
    return np.array([x, y, z], dtype=np.float32)

def cylinder_mesh(segments=40):
    theta = np.linspace(0, 2 * np.pi, segments, endpoint=False)
    vertices = []
    indices = []

    for t in theta:
        x = np.cos(t)
        y = np.sin(t)
        vertices.append((x, y, 0.0))
        vertices.append((x, y, 1.0))

    for i in range(segments):
        i0 = 2 * i
        i1 = 2 * i + 1
        i2 = 2 * ((i + 1) % segments)
        i3 = i2 + 1
        indices.extend([i0, i2, i1])
        indices.extend([i1, i2, i3])

    return np.array(vertices, dtype=np.float32), np.array(indices, dtype=np.int32)

def load_and_crop_background(path, size, rng):
    img = Image.open(path).convert("RGB")

    target_w, target_h = size
    target_aspect = target_w / target_h

    w, h = img.size
    img_aspect = w / h

    if img_aspect > target_aspect:
        # för bred -> crop i bredd
        new_h = h
        new_w = int(h * target_aspect)
    else:
        # för hög -> crop i höjd
        new_w = w
        new_h = int(w / target_aspect)

    max_x = max(0, w - new_w)
    max_y = max(0, h - new_h)

    x0 = int(rng.uniform(0, max_x)) if max_x > 0 else 0
    y0 = int(rng.uniform(0, max_y)) if max_y > 0 else 0

    crop = img.crop((x0, y0, x0 + new_w, y0 + new_h))
    crop = crop.resize((target_w, target_h), Image.Resampling.LANCZOS)
    return crop

# =========================================================
# Bakgrundsbilder
# =========================================================
bg_paths = []
bg_paths.extend(glob.glob(str(BACKGROUND_DIR / "*.jpg")))
bg_paths.extend(glob.glob(str(BACKGROUND_DIR / "*.jpeg")))
bg_paths.extend(glob.glob(str(BACKGROUND_DIR / "*.png")))

if not bg_paths:
    raise FileNotFoundError("Hittade inga bakgrundsbilder i mappen 'backgrounds'.")

# =========================================================
# OpenGL setup
# =========================================================
ctx = moderngl.create_standalone_context()
ctx.enable(moderngl.DEPTH_TEST)

color_tex = ctx.texture((W, H), components=4)
depth_rb = ctx.depth_renderbuffer((W, H))
fbo = ctx.framebuffer(color_attachments=[color_tex], depth_attachment=depth_rb)

cyl_prog = ctx.program(
    vertex_shader="""
    #version 330
    in vec3 in_pos;
    uniform mat4 M;
    uniform mat4 V;
    uniform mat4 P;
    void main() {
        gl_Position = P * V * M * vec4(in_pos, 1.0);
    }
    """,
    fragment_shader="""
    #version 330
    uniform vec3 color;
    out vec4 f_color;
    void main() {
        f_color = vec4(color, 1.0);
    }
    """
)

bg_prog = ctx.program(
    vertex_shader="""
    #version 330
    in vec2 in_pos;
    in vec2 in_uv;
    out vec2 v_uv;
    void main() {
        v_uv = in_uv;
        gl_Position = vec4(in_pos, 0.0, 1.0);
    }
    """,
    fragment_shader="""
    #version 330
    uniform sampler2D bg_tex;
    in vec2 v_uv;
    out vec4 f_color;
    void main() {
        f_color = texture(bg_tex, v_uv);
    }
    """
)

bg_quad = np.array([
    -1.0, -1.0, 0.0, 0.0,
     1.0, -1.0, 1.0, 0.0,
    -1.0,  1.0, 0.0, 1.0,
     1.0,  1.0, 1.0, 1.0,
], dtype=np.float32)
bg_vbo = ctx.buffer(bg_quad.tobytes())
bg_vao = ctx.vertex_array(bg_prog, [(bg_vbo, "2f 2f", "in_pos", "in_uv")])

vertices, indices = cylinder_mesh(segments=40)
vbo = ctx.buffer(vertices.tobytes())
ibo = ctx.buffer(indices.tobytes())
vao = ctx.vertex_array(cyl_prog, [(vbo, "3f", "in_pos")], ibo)

colors = np.array([
    [1.0, 0.2, 0.2],
    [0.2, 0.4, 1.0],
    [0.2, 0.8, 0.3],
    [1.0, 0.6, 0.1],
    [0.6, 0.2, 0.8],
    [0.1, 0.9, 0.9],
    [1.0, 0.2, 0.8],
    [0.9, 0.9, 0.2],
    [0.5, 0.3, 0.1],
    [1.0, 0.5, 0.7],
], dtype=np.float32)

# =========================================================
# Root-manifest
# =========================================================
manifest = {
    "run_name": run_name,
    "base_dir": str(BASE_DIR),
    "n_scenes": N_SCENES,
    "images_per_scene": IMAGES_PER_SCENE,
    "background_dir": str(BACKGROUND_DIR),
    "scenes": []
}

# =========================================================
# Huvudloop: många scener
# =========================================================
for scene_idx in range(N_SCENES):
    scene_name = f"scene_{scene_idx:04d}"
    scene_dir = BASE_DIR / scene_name
    image_dir = scene_dir / "images"
    scene_dir.mkdir(parents=True, exist_ok=True)
    image_dir.mkdir(parents=True, exist_ok=True)

    # Skapa scen
    cylinders, scene_data = generate_scene_cylinders(rng)

    # Spara scene.json
    with open(scene_dir / "scene.json", "w", encoding="utf-8") as f:
        json.dump(scene_data, f, indent=2)

    height_max = max(h for *_xyz, h in cylinders) if cylinders else 1.0
    scene_center = np.array([0.0, 0.0, height_max * 0.5], dtype=np.float32)

    scene_width = X_RANGE[1] - X_RANGE[0]
    scene_depth = Y_RANGE[1] - Y_RANGE[0]
    scene_diagonal = np.sqrt(scene_width**2 + scene_depth**2 + height_max**2)

    min_cam_radius = MIN_CAM_RADIUS_FACTOR * scene_diagonal
    max_cam_radius = MAX_CAM_RADIUS_FACTOR * scene_diagonal

    camera_matrices = []

    for img_idx in range(IMAGES_PER_SCENE):
        elev = float(rng.uniform(*ELEV_RANGE))
        azim = float(rng.uniform(0, 360))
        cam_radius = float(rng.uniform(min_cam_radius, max_cam_radius))

        eye = camera_position(cam_radius, elev, azim)
        target = scene_center
        up = np.array([0.0, 0.0, 1.0], dtype=np.float32)

        V = look_at(eye, target, up)
        P = perspective(35.0, W / H, 0.1, 100.0)

        # Bakgrund med slumpad crop
        bg_path = rng.choice(bg_paths)
        bg_img = load_and_crop_background(bg_path, (W, H), rng)
        bg_arr = np.array(bg_img, dtype=np.uint8)

        bg_tex = ctx.texture((W, H), 3, data=bg_arr.tobytes())
        bg_tex.filter = (moderngl.LINEAR, moderngl.LINEAR)

        # Rendera bakgrunden
        fbo.use()
        ctx.clear(0.0, 0.0, 0.0, 1.0, depth=1.0)

        ctx.disable(moderngl.DEPTH_TEST)
        bg_tex.use(location=0)
        bg_prog["bg_tex"].value = 0
        bg_vao.render(moderngl.TRIANGLE_STRIP)

        # Rendera cylindrar ovanpå
        ctx.enable(moderngl.DEPTH_TEST)

        cyl_prog["V"].write(V.T.tobytes())
        cyl_prog["P"].write(P.T.tobytes())

        for j, (x0, y0, r, h) in enumerate(cylinders):
            M = np.array([
                [r,   0.0, 0.0, x0],
                [0.0, r,   0.0, y0],
                [0.0, 0.0, h,   0.0],
                [0.0, 0.0, 0.0, 1.0]
            ], dtype=np.float32)

            cyl_prog["M"].write(M.T.tobytes())
            cyl_prog["color"].value = tuple(colors[j % len(colors)])
            vao.render()

        data = fbo.read(components=4, alignment=1)
        img = Image.frombytes("RGBA", (W, H), data).transpose(Image.FLIP_TOP_BOTTOM)

        filename = f"frame_{img_idx:03d}.png"
        img.save(image_dir / filename)

        camera_matrices.append(V.tolist())

        bg_tex.release()

    # Spara bara kameramatriserna i JSON
    with open(scene_dir / "camera_matrices.json", "w", encoding="utf-8") as f:
        json.dump(camera_matrices, f, indent=2)

    manifest["scenes"].append({
        "scene_name": scene_name,
        "scene_file": "scene.json",
        "camera_file": "camera_matrices.json",
        "image_dir": "images",
        "n_images": IMAGES_PER_SCENE,
        "n_cylinders": len(cylinders)
    })

    print(f"Klart: {scene_name} med {len(cylinders)} cylindrar")

# Spara root-manifest
with open(BASE_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"\nAllt klart. Dataset sparat i: {BASE_DIR}")

: 